# 04 · OCR — PaddleOCR ⇒ VietOCR

Hộp trong sơ đồ offline: **`OCR (PaddleOCR ⇒ VietOCR) → ocr_result`**

Hai tầng: PaddleOCR **chỉ khoanh vùng** (`det=True, rec=False`), VietOCR
`vgg_transformer` **đọc chữ**. Không dùng recognizer `lang="vi"` của Paddle — đó là model
đa ngôn ngữ hệ Latin, sai dấu tiếng Việt (đo được: 31% so với 44%).

Chia bốn vùng theo chiều cao: `top_bar` < 0.20 · `mid_frame` 0.20–0.80 ·
`subtitle_bar` 0.80–0.92 · `news_ticker` ≥ 0.92 (chuẩn EBU R95).

> ⚠️ **Kiểm `det_limit_side_len`.** PaddleOCR mặc định thu cạnh dài về 960 trước khi dò —
> đúng thứ làm OCR tụt 44% → 34% khi đo trên R1-17. Nếu ô cài đặt chưa có tham số này,
> thêm `det_limit_side_len=1920, det_limit_type="max"`.

> **Notebook này tự chạy đủ chuỗi**: cắt keyframe (**cách 2 — lấy mẫu theo tốc độ**) →
> dựng index BEiT3 + CLIP → chạy và đánh giá model. Không cần chạy notebook nào trước.
>
> **Cần `fps_map.json`** ở `/content/drive/MyDrive/aic26/fps_map.json` — chép từ
> `data_raw/fps_map.json` trong repo lên đó trước khi chạy. Cách 2 không dò fps từ
> video; đó là số chốt của BTC, dùng để đánh số frame đáp án.

**Người phụ trách: Vanh**


> **Chạy trong một phiên Colab RIÊNG.** Thư viện giữa các notebook xung đột nhau
> (paddleocr/vietocr hạ cấp Pillow), nên đừng chạy hai notebook trong cùng một runtime.

**Sửa đúng một ô: `SECTION 02a`.** Bốn thứ: `PERSON` · `VERSION` · `VIDEO_LIST` ·
`MAX_VIDEOS`. Mọi đường dẫn tự suy ra từ đó.

Mỗi người làm việc độc lập trên thư mục riêng của mình, nên **không cần phối hợp chọn
video với nhau** — trùng video cũng không sao, đầu ra nằm ở hai chỗ khác nhau.

```
/content/drive/MyDrive/aic26/
├── fps_map.json                     ← dùng chung, chép từ data_raw/ trong repo
├── transnetv2-pytorch-weights.pth   ← dùng chung, tự tải lần đầu
└── output_demo/
    ├── Vanh/                        ← mỗi người một nhánh
    │   ├── v001/                    ← mỗi cấu hình một version
    │   │   ├── progress.json
    │   │   ├── zips/      AIC2026_frames_001.zip …
    │   │   └── indexes/   beit3.index  clip.index  keyframe_metadata.json
    │   │                  keyframes_list.json  image_mapping.json  fps_map.json
    │   └── v002/                    ← lần chạy với cấu hình khác
    ├── Phat/v001/
    ├── An/v001/
    └── Bang/v001/
```

**Khi nào tăng `VERSION`:** đổi cách cắt · đổi tham số lấy mẫu · đổi model encoder ·
đổi độ phân giải lưu ảnh. Chỉ chạy thêm video vào cùng cấu hình thì **giữ nguyên** —
`progress.json` tự bỏ qua video đã xong.

Ô config sẽ cảnh báo nếu version đang chọn đã có dữ liệu, để bạn không lỡ trộn hai
cấu hình vào cùng một index.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02a – AI LÀM PHẦN NÀO + PHẠM VI TEST   ← Ô DUY NHẤT CẦN SỬA
# ═══════════════════════════════════════════════════════════════════════════════
# %%

# ─── 1. Bạn là ai ────────────────────────────────────────────────────────────
# Đổi đúng tên mình. Mỗi người ghi vào thư mục riêng nên không giẫm chân nhau.
PERSON = "Vanh"          # "Vanh" · "Phat" · "An" · "Bang"

# ─── 2. Lần chạy thứ mấy ─────────────────────────────────────────────────────
# Mỗi lần đổi CÁCH LÀM thì tăng version, đừng ghi đè lên lần trước — có giữ lại
# mới so được hai lần chạy với nhau bằng notebook chấm điểm.
#
# Tăng version khi:
#   · đổi cách cắt          (cách 1 ↔ cách 2)
#   · đổi tham số lấy mẫu   (SECONDS_PER_EXTRA, N_MAX, MARGIN…)
#   · đổi model encoder     (BEiT3 ↔ SigLIP…)
#   · đổi độ phân giải lưu ảnh
#
# KHÔNG cần tăng khi chỉ chạy thêm video vào cùng một cấu hình — progress.json
# tự bỏ qua video đã xong, chạy tiếp là được.
VERSION = "v001"        # "v001" · "v002" · "v003" …

# ─── 3. Chạy video nào ───────────────────────────────────────────────────────
# Liệt kê tên video muốn chạy. Để TRỐNG = lấy từ đầu danh sách TEST_VIDEOS.
VIDEO_LIST = [
    # "L21_V015",   # R1-1  KIS · 4 phi hành gia — video đã đo kỹ ở Q1
    # "L30_V092",   # R1-17 KIS · phông đỏ COVID-19 — video để thử OCR
]

# Giới hạn số video. 0 = không giới hạn.
# Lần đầu để 3 cho nhanh (~30 phút); chạy thông hết chuỗi rồi mới nâng lên.
MAX_VIDEOS = 3

assert VERSION.startswith("v") and VERSION[1:].isdigit(), \
    f'VERSION phải dạng "v001", "v002"… — đang là "{VERSION}"'

# ─── 4. Đường dẫn suy ra từ version ──────────────────────────────────────────
# CẢ NHÓM DÙNG CHUNG một thư mục — cùng chỗ mà cut_frames_1…5 ghi keyframe ra.
#
#   aic26/Frame_Transnet_2026_<version>/
#     zips/       AIC2026_frames_p1_001.zip ...
#     indexes/    beit3.index  clip.index  keyframe_metadata.json
#                 keyframes_list.json  image_mapping.json  fps_map.json
#     features/   ocr/  asr/  caption/  tag/  object/
#     progress_p1.json … progress_p5.json     (do bước cắt ghi)
#     progress_<Tên>.json                     (do notebook này ghi)
#
# PERSON chỉ dùng để tách file tiến độ, không tách thư mục nữa — nếu tách thì
# mỗi người có một index riêng, mà index phải là một khối duy nhất cho toàn bộ
# dữ liệu mới tìm được xuyên video.
DRIVE_OUT  = f"/content/drive/MyDrive/aic26/Frame_Transnet_2026_{VERSION}"

# Dùng chung, nằm NGOÀI thư mục riêng để 4 người khỏi tải trọng số 4 lần
SHARED_DIR = "/content/drive/MyDrive/aic26"

print(f"[Person]   {PERSON}")
print(f"[Version]  {VERSION}")
print(f"[Output]   {DRIVE_OUT}")
print(f"[Videos]   " + (f"custom {len(VIDEO_LIST)}" if VIDEO_LIST else "head of list")
      + (f" · limit {MAX_VIDEOS}" if MAX_VIDEOS else " · no limit"))

# Cảnh báo nếu thư mục đã có dữ liệu — tránh trộn hai cấu hình vào cùng một chỗ
import os as _os
if _os.path.exists(f"{DRIVE_OUT}/progress.json"):
    import json as _json
    with open(f"{DRIVE_OUT}/progress.json") as _f:
        _p = _json.load(_f)
    _n = len(_p.get("done_videos", []))
    if _n:
        print()
        print(f"[!] {VERSION} already has {_n} videos cut and {len(_p.get('completed_zips', []))} zips.")
        print("    Continuing will skip those - fine if you are only adding videos.")
        print(f"    If you CHANGED THE METHOD, bump VERSION - do not mix two configs")
        print( "    into one index.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install Dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install] Installing baseline dependencies...")
_pip("--upgrade", "transnetv2-pytorch==1.0.5", "ffmpeg-python")
_pip("open_clip_torch")
_pip("faiss-cpu")   # faiss-gpu trên PyPI dừng ở 2022, không cài được
_pip("tqdm")
_pip("transformers>=4.40.0", "Pillow", "opencv-python-headless")
_pip("gdown")
# BEiT3 (unilm) — dependency thật, verify bằng cách tự cài + test trong sandbox
# trước khi đưa vào đây. deepspeed KHÔNG cần cho inference (verify: chỉ dùng
# trong run_beit3_finetuning.py, không import trong modeling_finetune.py).
_pip("torchscale==0.2.0", "timm==0.4.12", "torchmetrics==0.7.3")
_pip("tensorboardX", "ftfy", "blobfile", "sentencepiece")
# setuptools<81: torchmetrics 0.7.3 dùng pkg_resources (deprecated, bị xóa
# khỏi setuptools>=81) — verify lỗi thật này bằng cách tự chạy import và bắt
# ModuleNotFoundError trong sandbox trước khi thêm dòng pin này.
_pip("setuptools<81")
print("[Install] All done\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Install + Config
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import subprocess, sys, gc, os, json, glob, zipfile, shutil, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path
from PIL import Image, ImageDraw
from collections import defaultdict, Counter
import textwrap

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
def _uninstall(*args):
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-q", "-y", *args])

# Uninstall conflicting paddle versions trước
_uninstall("paddlepaddle-gpu", "paddlepaddle", "paddleocr")

print("[Install] Installing dependencies...")
_pip("open_clip_torch", "transformers>=4.40.0", "opencv-python-headless")
_pip("paddlepaddle")        # CPU – đủ cho OCR
_pip("paddleocr<3.0")       # pin 2.x API, stable — dùng CHỈ để detect box
_pip("vietocr")             # recognition riêng cho tiếng Việt — xem Section 03
_pip("ultralytics")
_pip("einops", "timm")

# ── Fix: broken/mixed Pillow install (ImportError: is_directory) ──────────────
# Root cause THẬT: một trong các package vừa cài (paddlepaddle/paddleocr/
# vietocr) làm Pillow bị cài LẪN giữa 2 version — PIL/ImageFont.py từ 1 bản,
# PIL/_util.py từ bản khác → ImageFont.py import is_directory nhưng _util.py
# hiện tại không có hàm đó.
# Fix trước (Pillow<10, theo running.py) SAI ở đây vì 2 lý do:
#   1. running.py gốc dùng repo fork riêng (bmd1905/vietnamese-ocr, clone qua
#      git), KHÁC với package `vietocr` chuẩn trên PyPI đang dùng ở đây —
#      constraint Pillow<10 của họ không transfer đúng sang pipeline này.
#   2. Colab hiện tại chạy Python 3.12 — Pillow<10 có thể không có wheel sẵn
#      cho 3.12, khiến "pip install" trả về thành công nhưng cài ra bản hỏng.
# Fix đúng: uninstall sạch rồi force-reinstall 1 bản Pillow DUY NHẤT (không ép
# version cũ), làm bước CUỐI trong Section 00 để không package nào sau đó
# ghi đè/làm lẫn lại.
_uninstall("Pillow", "pillow")
_pip("--force-reinstall", "--no-cache-dir", "Pillow")
print("[Install] Done\n")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Eval] Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"[Eval] VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

DRIVE_MOUNT = "/content/drive"
if not os.path.ismount(DRIVE_MOUNT):
    from google.colab import drive; drive.mount(DRIVE_MOUNT)

OUTPUT_ROOT  = DRIVE_OUT
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"   # ← Part 2 Section 13b dùng path này cho ASR
INDEX_DIR    = f"{OUTPUT_ROOT}/indexes"
ZIP_DIR      = f"{OUTPUT_ROOT}/zips"
EVAL_DIR     = "/content/eval_frames"
EVAL_OUT     = "/content/eval_outputs"
shutil.rmtree(EVAL_DIR, ignore_errors=True)   # xóa stale files từ crashed runs
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
Path(EVAL_OUT).mkdir(exist_ok=True)

N_EVAL_FRAMES = 2000   # ← tăng từ 300 lên 2000

def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()

# ── Vietnamese-compatible font (fix □ tofu boxes in matplotlib text) ───────────
# Root cause: DejaVu Sans Mono (matplotlib's default "monospace" family) lacks
# glyphs for precomposed Vietnamese vowels with combining diacritics. Noto Sans
# has full Latin Extended Additional coverage (which Vietnamese diacritics use).
# Installed via apt (fonts-noto-core) — more reliable on Colab than a raw URL.
try:
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-core"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=60)
    import matplotlib.font_manager as _fm
    _noto = glob.glob("/usr/share/fonts/**/NotoSans-Regular.ttf", recursive=True)
    if _noto:
        _fm.fontManager.addfont(_noto[0])
        _VN_FONT = _fm.FontProperties(fname=_noto[0]).get_name()
        plt.rcParams["font.family"] = _VN_FONT
        print(f"[Font] Vietnamese-compatible font registered: {_VN_FONT} ({_noto[0]})")
    else:
        _VN_FONT = "DejaVu Sans"
        print("[Font] WARNING: Noto Sans not found, falling back to DejaVu Sans")
except Exception as e:
    _VN_FONT = "DejaVu Sans"
    print(f"[Font] WARNING: font install failed ({e}), falling back to DejaVu Sans")

# ── JSON results helper (dùng chung cho mọi section) ───────────────────────────
def save_json_results(name: str, data: dict) -> str:
    """Lưu {frame_path_or_name: result} → EVAL_OUT/{name}_results.json"""
    path = os.path.join(EVAL_OUT, f"{name}_results.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    print(f"  [JSON] {name}: {len(data)} entries → {path}")
    return path

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Mount Drive  ← PHẢI mount TRƯỚC mọi thứ liên quan /content/drive
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import os
DRIVE_MOUNT = "/content/drive"
if os.path.ismount(DRIVE_MOUNT):
    print("[Drive] Already mounted.")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted OK.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02 – Config & Dirs  ← SAU khi Drive đã mount
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import sys, json, glob, time, math, zipfile, shutil, subprocess as _sp
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ─── MODE ────────────────────────────────────────────────────────────────────
# "TEST"  → chỉ xử lý TEST_VIDEOS (~83 videos, đủ để kiểm thử pipeline)
# "FULL"  → toàn bộ ~1478 videos (production run)
MODE = "TEST"   # ← đổi thành "FULL" khi cần chạy thật

# ─── PATHS (path cứng, không đổi) ────────────────────────────────────────────
VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = DRIVE_OUT
CKPT_TNV2    = f"{SHARED_DIR}/transnetv2-pytorch-weights.pth"
LOCAL_KF_DIR = "/content/kf_buffer"

# ─── TEST VIDEOS ──────────────────────────────────────────────────────────────
# Đối chiếu lại với All_Queries.docx (bản đầy đủ, verify từng câu qua raw XML,
# không dựa vào comment cũ). Round 1 (25 câu) + Round 2 (30 câu) đã full fill,
# Round 3 CHƯA có nội dung trong docx này (chỉ có tiêu đề, không có query nào).
#
# Round 1 GT videos — verify từng câu, KHỚP 100% với docx (24/25 câu có video;
# R1-21 không có video xác nhận — thay vào đó dùng K05_V006 làm benchmark thay
# thế, đúng như docx ghi chú: "Vid gần nhất giống mô tả... K05_V006 (khác mỗi
# ở chỗ không phải "bọ" mà là "chim")... nếu muốn lấy benchmark cho câu này
# thì lấy vid đó làm truth value")
_GT_L = {
    "L21_V015",  # R1-1 KIS: 4 phi hành gia mặc áo đen / tàu vũ trụ tư nhân
    "L21_V027",  # R1-14 KIS: điêu khắc cát, khói hồng
    "L21_V029",  # R1-2 KIS: đàn hổ con quý hiếm miền Nam
    "L22_V022",  # R1-23 KIS: thị trấn ven biển / cá mập (phim Spielberg 1975)
    "L22_V030",  # R1-8 KIS: lễ hội ẩm thực Nhật, cô bé đeo bạch tuộc đỏ
    "L23_V007",  # R1-24 KIS: đua xe đạp, 3 tay đua đồng phục trắng-vàng-xanh
    "L23_V017",  # R1-25 KIS: flycam đua xe đạp, áo xanh dương trắng vượt lên
    "L24_V033",  # R1-16 TRAKE: múa lân vàng đen trắng / lân quay cột
    "L25_V058",  # R2-21 QA: bài tập vận dụng, khoảng cách AB 2 chữ số, đáp C
    "L26_V004",  # R1-20 KIS: panna cotta 3 ly / nho đỏ bạc hà
    "L26_V056",  # R1-6 KIS: gỏi cuốn chay, bánh tráng vàng tím (video đúng)
    "L26_V072",  # R1-18 TRAKE: cắt nấm → củ năng → đậu hũ → bếp
    "L26_V178",  # R1-22 QA: bánh ít trần, 200g thịt nạc xay
    "L26_V194",  # R1-4 TRAKE: bột → măng tây → dầu → chiên
    "L26_V200",  # R1-12 KIS: trang trí bánh rán, chocolate + chuối + dâu
    "L26_V385",  # R1-6 KIS: gỏi cuốn chay (video tìm được, dù L26_V056 đúng hơn)
    "L27_V010",  # R1-19 QA: đình thần Nguyễn Trung Trực Kiên Giang / 2 câu thơ
    "L27_V013",  # R1-9 KIS: thu hoạch dứa miền Tây, bà cụ + cô gái áo hồng + ghe xanh
    "L27_V014",  # R1-5 KIS: cho dê ăn, áo thun trắng + áo kẻ sọc tím
    "L29_V023",  # R1-7 KIS: chim đầu đen ánh xanh mắt đỏ, rừng Nam Bộ
    "L30_V014",  # R2-11 KIS: phụ nữ lớn tuổi đọc sách / bìa ảnh cụ lão
    "L30_V017",  # R1-10 KIS: 3 người chơi nhạc cụ kim loại tròn / kệ sách
    "L30_V040",  # R2-18 KIS: mèo trong phòng, tô kim loại úp, bảng tên "Nà Ní"
    "L30_V057",  # R1-11 KIS: mảnh bìa đổ bóng thành chân dung / mũ lưỡi trai đen
    "L30_V072",  # R1-15 QA: câu lạc bộ FANA trao quà Xã Giang Ly Khánh Hòa
    "L30_V092",  # R1-17 KIS: trao quà từ thiện bệnh viện, phông đỏ COVID-19
    "L30_V095",  # R1-13 KIS: vệ sinh máy ảnh, ống kính + khăn tím hồng + tăm bông
}

# Round 2 GT videos (K-series = dataset batch K). Verify 100% với docx —
# 27/30 câu có video xác nhận (R2-16, R2-19 explicit "chưa tìm được đáp án"
# trong docx → KHÔNG có GT, cố tình bỏ qua. R2-12, R2-23 KHÔNG có video ID
# xác nhận trong docx dù trước đây từng nhớ nhầm là có → xem _LEGACY_UNCONFIRMED)
_GT_K = {
    "K01_V009",  # R2-4 KIS: cứu hộ lũ, cụ bà đội nón lá trên xuồng cam
    "K01_V018",  # R2-15 KIS: cô gái phỏng vấn + Deadpool + múa đao
    "K02_V005",  # R2-5 KIS + R2-22 TRAKE: robot vẽ tranh + sách MILK / khám xét
    "K03_V019",  # R2-1 KIS: nhiều người áo cờ đỏ sao vàng trước biểu tượng con cua
    "K03_V022",  # R2-29 KIS: cảng biển, cầu dài ra biển, người đội nón tai bèo
    "K03_V023",  # R2-8 KIS: cầu vòm thép đỏ → ca nô vệt sóng tròn
    "K05_V013",  # R2-24 KIS: ngư dân Campuchia nâng cá lớn nguy cấp
    "K05_V018",  # R2-25 KIS: robot 4 chân bánh xe trên đồng cỏ + bò
    "K05_V025",  # R2-7 KIS: người tuyết khổng lồ đội mũ + bình gốm băng tuyết
    "K06_V010",  # R2-20 KIS: cào muối đồng, đoàn người, "Gừng cay muối mặn"
    "K07_V025",  # R2-10 TRAKE: xưởng lắp ráp robot + công nhân + cửa ô tô
    "K07_V030",  # R2-17 KIS: phụ nữ đội nón len đeo kính "Happy New Year"
    "K09_V007",  # R2-30 KIS: kị sĩ cưỡi ngựa đồng cỏ xanh / cờ đỏ
    "K09_V012",  # R2-26 KIS: phỏng vấn Trump + đoàn người ven biển vác bao trắng
    "K09_V015",  # R2-13 KIS: bánh giá 9€ và 4.80€ / sản xuất 12 cái/tầng
    "K09_V018",  # R2-6 KIS: đài phun nước trăm ngọn nến / phụ nữ bên ly nến
    "K12_V001",  # R2-9 KIS: cá nhám 211kg treo trên xe cẩu
    "K14_V027",  # R2-2 KIS: lễ hội đèn lồng, hanbok, đèn lồng khổng lồ hình người
    "K15_V010",  # R2-27 KIS: họp giá điện, màn hình LED chữ trắng nền xanh
    "K16_V010",  # R2-14 KIS: icon hình não 5×10 / thống kê tác dụng phụ thuốc
    "K17_V003",  # R2-3 QA: bản đồ Việt Nam trái cây / Khu Du Lịch Quốc Gia Mộc Châu
    "K18_V002",  # R2-28 KIS: nhiều loài cá cảnh, cá đĩa, cá rồng, cá la hán
    "K05_V006",  # R1-21 BENCHMARK THAY THẾ (không phải GT gốc): docx ghi rõ
                 # câu gốc "năm ngoái BTC nói lỗi", K05_V006 là video gần nhất
                 # (khác "bọ" → "chim") được docx đề xuất dùng làm truth value
                 # nếu cần benchmark cho câu này.
}

# ⚠ LEGACY UNCONFIRMED — trước đây từng coi là GT nhưng All_Queries.docx (bản
# đầy đủ, verify qua raw XML) KHÔNG có video ID cho các câu này:
#   K04_V013 (từng gán cho R2-23) — R2-23 trong docx mới KHÔNG có dòng
#     "Kxx_Vxxx,timestamp" nào, chỉ có mô tả + nhận xét, không có đáp án.
#   K07_V019 (từng gán cho R2-12) — tương tự, R2-12 không có video ID.
#   L26_V176, L29_V020 (từng gán cho R3-1, R3-2) — docx này DỪNG NGAY tại
#     tiêu đề "Round 3", không có nội dung Round 3 nào cả, nên không thể
#     verify lại 2 video này từ nguồn hiện tại.
# Quyết định: KHÔNG xóa (đã cắt sẵn trong 95 video hiện có, xóa cũng không
# undo được), nhưng KHÔNG coi là GT đã verify — giữ trong diversity pool.
_LEGACY_UNCONFIRMED = {
    "K04_V013", "K07_V019", "L26_V176", "L29_V020",
}

# Diversity videos (NON-GT) — mở rộng để đạt ~25-30GB frame output (yêu cầu),
# thay vì chỉ 1-3 video/batch như trước (→ 8GB/95 video). Mở rộng lên
# K01-K18 × V001-V008 (144 video) + L21-L30 × V001-V010 (100 video) = 244 mới.
# Ước tính: 8GB/95 video ≈ 86MB frame-output/video trung bình → ~300 video
# tổng (GT+diversity) × 86MB ≈ 25.8GB, khớp mục tiêu 25-30GB. Video nào không
# tồn tại trên Drive sẽ tự động bị skip + WARN (đã có sẵn cơ chế ở Section 03).
_DIVERSITY = {
    "K01_V001", "K01_V002", "K01_V003", "K01_V004", "K01_V005", "K01_V006", "K01_V007", "K01_V008",
    "K02_V001", "K02_V002", "K02_V003", "K02_V004", "K02_V005", "K02_V006", "K02_V007", "K02_V008",
    "K03_V001", "K03_V002", "K03_V003", "K03_V004", "K03_V005", "K03_V006", "K03_V007", "K03_V008",
    "K04_V001", "K04_V002", "K04_V003", "K04_V004", "K04_V005", "K04_V006", "K04_V007", "K04_V008",
    "K05_V001", "K05_V002", "K05_V003", "K05_V004", "K05_V005", "K05_V006", "K05_V007", "K05_V008",
    "K06_V001", "K06_V002", "K06_V003", "K06_V004", "K06_V005", "K06_V006", "K06_V007", "K06_V008",
    "K07_V001", "K07_V002", "K07_V003", "K07_V004", "K07_V005", "K07_V006", "K07_V007", "K07_V008",
    "K08_V001", "K08_V002", "K08_V003", "K08_V004", "K08_V005", "K08_V006", "K08_V007", "K08_V008",
    "K09_V001", "K09_V002", "K09_V003", "K09_V004", "K09_V005", "K09_V006", "K09_V007", "K09_V008",
    "K10_V001", "K10_V002", "K10_V003", "K10_V004", "K10_V005", "K10_V006", "K10_V007", "K10_V008",
    "K11_V001", "K11_V002", "K11_V003", "K11_V004", "K11_V005", "K11_V006", "K11_V007", "K11_V008",
    "K12_V001", "K12_V002", "K12_V003", "K12_V004", "K12_V005", "K12_V006", "K12_V007", "K12_V008",
    "K13_V001", "K13_V002", "K13_V003", "K13_V004", "K13_V005", "K13_V006", "K13_V007", "K13_V008",
    "K14_V001", "K14_V002", "K14_V003", "K14_V004", "K14_V005", "K14_V006", "K14_V007", "K14_V008",
    "K15_V001", "K15_V002", "K15_V003", "K15_V004", "K15_V005", "K15_V006", "K15_V007", "K15_V008",
    "K16_V001", "K16_V002", "K16_V003", "K16_V004", "K16_V005", "K16_V006", "K16_V007", "K16_V008",
    "K17_V001", "K17_V002", "K17_V003", "K17_V004", "K17_V005", "K17_V006", "K17_V007", "K17_V008",
    "K18_V001", "K18_V002", "K18_V003", "K18_V004", "K18_V005", "K18_V006", "K18_V007", "K18_V008",
    "L21_V001", "L21_V002", "L21_V003", "L21_V004", "L21_V005", "L21_V006", "L21_V007", "L21_V008", "L21_V009", "L21_V010",
    "L22_V001", "L22_V002", "L22_V003", "L22_V004", "L22_V005", "L22_V006", "L22_V007", "L22_V008", "L22_V009", "L22_V010",
    "L23_V001", "L23_V002", "L23_V003", "L23_V004", "L23_V005", "L23_V006", "L23_V007", "L23_V008", "L23_V009", "L23_V010",
    "L24_V001", "L24_V002", "L24_V003", "L24_V004", "L24_V005", "L24_V006", "L24_V007", "L24_V008", "L24_V009", "L24_V010",
    "L25_V001", "L25_V002", "L25_V003", "L25_V004", "L25_V005", "L25_V006", "L25_V007", "L25_V008", "L25_V009", "L25_V010",
    "L26_V001", "L26_V002", "L26_V003", "L26_V004", "L26_V005", "L26_V006", "L26_V007", "L26_V008", "L26_V009", "L26_V010",
    "L27_V001", "L27_V002", "L27_V003", "L27_V004", "L27_V005", "L27_V006", "L27_V007", "L27_V008", "L27_V009", "L27_V010",
    "L28_V001", "L28_V002", "L28_V003", "L28_V004", "L28_V005", "L28_V006", "L28_V007", "L28_V008", "L28_V009", "L28_V010",
    "L29_V001", "L29_V002", "L29_V003", "L29_V004", "L29_V005", "L29_V006", "L29_V007", "L29_V008", "L29_V009", "L29_V010",
    "L30_V001", "L30_V002", "L30_V003", "L30_V004", "L30_V005", "L30_V006", "L30_V007", "L30_V008", "L30_V009", "L30_V010",
} | _LEGACY_UNCONFIRMED

# Hợp nhất: GT (~50 videos, verify 100% từ docx) + Diversity (~244+4 videos)
TEST_VIDEOS: set[str] = _GT_L | _GT_K | _DIVERSITY

print(f"[Mode] MODE = {MODE}")
print(f"[Mode] GT videos: {len(_GT_L)+len(_GT_K)} | Diversity: {len(_DIVERSITY)} | Total TEST set: {len(TEST_VIDEOS)}")

# ─── TRANSNETV2 ──────────────────────────────────────────────────────────────
TNV2_W        = 48
TNV2_H        = 27
CUT_THR       = 0.5
MIN_CUT_DIST  = 12
MIN_SCENE_LEN = 8
TNV2_CHUNK    = 200
TNV2_OVERLAP  = 25
TNV2_GDRIVE_ID = "1K7tpO6NMGVqCF30mwuyR-i2HZMA7Fsrg"

# ─── FRAME SELECTION (Paper Alg.1) ───────────────────────────────────────────
N_SAMPLE_PER_SCENE = 4
DEDUP_THR     = 0.90
JPEG_QUALITY  = 90

# ─── ZIP ─────────────────────────────────────────────────────────────────────
ZIP_EVERY_GB  = 2.0
ZIP_DIR       = f"{OUTPUT_ROOT}/zips"
INDEX_DIR     = f"{OUTPUT_ROOT}/indexes"
PROGRESS_FILE = f"{OUTPUT_ROOT}/progress_{PERSON}.json"   # tách theo người

# ─── MODELS ──────────────────────────────────────────────────────────────────
# BEiT3-large (Wang et al., CVPR 2023 — chính xác model [51] paper 2504.08384
# cite cho nhánh "fine-grained"). Checkpoint tốt nhất public cho retrieval:
# beit3_large_patch16_384_coco_retrieval.pth — 675M params, fine-tuned COCO
# retrieval, ITC dual-encoder. Verify TOÀN BỘ bằng cách tự clone source thật
# từ microsoft/unilm, cài dependency, patch lỗi torch._six, tải checkpoint
# thật (1.29GB) và chạy forward pass thành công (output 1024-dim, khớp
# encoder_embed_dim khai báo trong _get_large_config của chính unilm).
BEIT3_CKPT_URL   = "https://github.com/addf400/files/releases/download/beit3/beit3_large_patch16_384_coco_retrieval.pth"
BEIT3_SPM_URL    = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"
BEIT3_MODEL_NAME = "beit3_large_patch16_384_retrieval"
BEIT3_INPUT_SIZE = 384
BEIT3_BATCH      = 8   # 675M params fp32 — conservative batch cho T4
BEIT3_CODE_DIR   = "/content/unilm_beit3"   # nơi clone source code thật
# ViT-bigG-14 (LAION, laion2b_s39b_b160k) — giữ nguyên theo quyết định trước:
# nhánh "coarse-grained" dùng CLIP tốt nhất khả thi, KHÔNG cần đúng tên [41]
# (paper cite CLIP gốc OpenAI nhưng không chỉ rõ checkpoint, nên chọn bản
# LAION mạnh nhất fit T4 là hợp lý — đã thảo luận và quyết định trước đó).
CLIP_MODEL       = "ViT-bigG-14"
CLIP_PRETRAIN    = "laion2b_s39b_b160k"
CLIP_BATCH       = 16
FAISS_BEIT3_DIM  = 1024  # verify thật: encoder_embed_dim trong _get_large_config
FAISS_CLIP_DIM   = 1280  # bigG-14 embed_dim, verify từ model config chính thức
NEIGHBOR_WINDOW  = 3

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for d in [OUTPUT_ROOT, ZIP_DIR, INDEX_DIR, LOCAL_KF_DIR, SHARED_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"[Config] VIDEO_ROOT  : {VIDEO_ROOT}")
print(f"[Config] OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"[Config] LOCAL buffer: {LOCAL_KF_DIR}")
print(f"[Config] ZIP every   : {ZIP_EVERY_GB} GB")

# ── Load / Init progress ───────────────────────────────────────────────────────
def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            p = json.load(f)
        print(f"[Resume] {len(p.get('done_videos',[]))} videos done, "
              f"{p.get('zip_count',0)} zips created.")
        return p
    print("[Resume] No progress file found – starting fresh.")
    return {
        "done_videos": [],
        "zip_count": 0,
        "completed_zips": [],
        "buffer_videos": [],
        "buffer_size_bytes": 0,
        "total_frames_cut": 0,
        "index_done_beit3": [],
        "index_done_clip":   [],
        "index_phase_done":  False,
        # Feature extraction phase (Part 2)
        "done_ocr":     [],
        "done_asr":     [],
        "done_caption": [],
        "done_tags":    [],
        "done_objects": [],
        "features_merged": False,
        "last_updated": None,
    }

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02b – Buffer Recovery (quan trọng: phát hiện videos bị mất khi disconnect)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Khi Colab disconnect, /content/kf_buffer bị xóa.
# Videos đã mark done nhưng chưa kịp zip → keyframes mất.
# Section này phát hiện và đưa chúng về hàng đợi lại.
# %%

def _find_unzipped_videos() -> set[str]:
    """Tìm video names đã có trong các completed_zips."""
    found = set()
    for zname in PROGRESS.get("completed_zips", []):
        zpath = os.path.join(ZIP_DIR, zname)
        if not os.path.exists(zpath): continue
        with zipfile.ZipFile(zpath) as zf:
            for name in zf.namelist():
                parts = name.split("/")
                if parts: found.add(parts[0])
    return found

_zipped_videos = _find_unzipped_videos()
_buffer_lost   = [v for v in PROGRESS.get("buffer_videos", [])
                  if v not in _zipped_videos]

if _buffer_lost:
    print(f"[Recovery] Found {len(_buffer_lost)} buffered videos lost on disconnect:")
    for v in _buffer_lost:
        print(f"  ↩ {v}")
    # Đưa về done_videos để cut lại
    for v in _buffer_lost:
        if v in PROGRESS["done_videos"]:
            PROGRESS["done_videos"].remove(v)
    PROGRESS["buffer_videos"]    = [v for v in PROGRESS["buffer_videos"]
                                    if v not in _buffer_lost]
    PROGRESS["buffer_size_bytes"] = 0
    save_progress(PROGRESS)
    print("[Recovery] Re-queued. They will be re-cut in Section 07.")
else:
    print("[Recovery] Buffer OK - no videos lost.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03 – List Videos (filter theo MODE)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

def list_all_videos(root: str) -> list[str]:
    vids = sorted(set(
        glob.glob(os.path.join(root, "**", "*.mp4"), recursive=True) +
        glob.glob(os.path.join(root, "*.mp4"))
    ))
    if not vids:
        raise FileNotFoundError(f"No videos found in {root}")
    return vids

_all_raw      = list_all_videos(VIDEO_ROOT)
# Filter theo MODE
if MODE == "TEST":
    ALL_VIDEOS = [v for v in _all_raw if Path(v).stem in TEST_VIDEOS]
    _missing   = TEST_VIDEOS - {Path(v).stem for v in _all_raw}
    if _missing:
        print(f"[TEST] WARN: {len(_missing)} TEST_VIDEOS not found on Drive:")
        for m in sorted(_missing): print(f"  ✗ {m}")
else:  # FULL
    ALL_VIDEOS = _all_raw

# ─── Lọc theo VIDEO_LIST ở SECTION 02a ───────────────────────────────────────
if VIDEO_LIST:
    _wanted = set(VIDEO_LIST)
    ALL_VIDEOS = [v for v in ALL_VIDEOS if Path(v).stem in _wanted]
    _missing = _wanted - {Path(v).stem for v in ALL_VIDEOS}
    if _missing:
        print(f"[Select]   WARN not found on Drive: {sorted(_missing)}")
    print(f"[Select]   {len(ALL_VIDEOS)} videos from VIDEO_LIST")
else:
    ALL_VIDEOS = sorted(ALL_VIDEOS)
    print(f"[Select]   VIDEO_LIST empty -> taking from head of {len(ALL_VIDEOS)} videos")

done_set  = set(PROGRESS["done_videos"])
todo_list = [v for v in ALL_VIDEOS if Path(v).stem not in done_set]

if MAX_VIDEOS and len(todo_list) > MAX_VIDEOS:
    todo_list = todo_list[:MAX_VIDEOS]
    print(f"[Limit] Running only {MAX_VIDEOS} videos: {[Path(v).stem for v in todo_list]}")
    print( "           Set MAX_VIDEOS = 0 in SECTION 02a to run everything.")

total_gb = sum(os.path.getsize(v) for v in ALL_VIDEOS) / 1024**3
done_gb  = sum(os.path.getsize(v) for v in ALL_VIDEOS
               if Path(v).stem in done_set) / 1024**3

print(f"\n[Videos] MODE={MODE} | Total: {len(ALL_VIDEOS)} | Done: {len(done_set)} | Remaining: {len(todo_list)}")
print(f"[Size]   Total: {total_gb:.1f} GB | Done: {done_gb:.1f} GB")
print(f"[Sample] First 5 remaining:")
for v in todo_list[:5]:
    print(f"  {Path(v).stem}  ({os.path.getsize(v)/1024**2:.0f} MB)")

if not todo_list:
    print("\n[OK] All videos cut. Go to Section 09 to build the index.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 04 – Load TransNetV2 (auto-download weights nếu thiếu)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import torch
from transnetv2_pytorch import TransNetV2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[System] Device: {DEVICE}")

if not os.path.exists(CKPT_TNV2):
    print(f"[TransNetV2] Weights not found → downloading from GDrive...")
    import gdown
    gdown.download(id=TNV2_GDRIVE_ID, output=CKPT_TNV2, quiet=False)
    print(f"[TransNetV2] Downloaded → {CKPT_TNV2}")

def load_tnv2(ckpt_path: str):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m = TransNetV2()
    m.load_state_dict(state)
    return m.eval().to(DEVICE)

if todo_list:
    tnv2 = load_tnv2(CKPT_TNV2)
    print("[TransNetV2] Loaded OK")
else:
    print("[TransNetV2] Skipped (no videos remaining)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 06 – Helper Functions (giữ nguyên từ nb_01)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import ffmpeg, cv2
from PIL import Image
from tqdm import tqdm

# ──── Video Decode 48×27 cho TransNetV2 ────────────────────────────────────────
def decode_video_tnv2(path: str) -> np.ndarray:
    stream, _ = (
        ffmpeg.input(path)
        .output("pipe:", format="rawvideo", pix_fmt="rgb24", s=f"{TNV2_W}x{TNV2_H}")
        .run(capture_stdout=True, capture_stderr=True)
    )
    return np.frombuffer(stream, np.uint8).reshape([-1, TNV2_H, TNV2_W, 3])

def get_video_fps(path: str) -> float:
    try:
        import json as _json
        r = _sp.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                     "-show_entries", "stream=r_frame_rate", "-of", "json", path],
                    capture_output=True, text=True, timeout=15)
        info = _json.loads(r.stdout)
        fr = info["streams"][0]["r_frame_rate"]
        n, d = fr.split("/"); return float(n) / float(d)
    except Exception:
        cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release(); return fps

# ──── TransNetV2 Chunked (fix OOM cho video dài) ────────────────────────────────
def run_tnv2_chunked(frames: np.ndarray, model,
                     chunk: int = TNV2_CHUNK, ovl: int = TNV2_OVERLAP) -> np.ndarray:
    n = frames.shape[0]
    preds  = np.zeros(n, dtype=np.float32)
    counts = np.zeros(n, dtype=np.int32)
    start  = 0
    while start < n:
        end = min(start + chunk, n)
        t   = torch.from_numpy(frames[start:end].copy()).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            sp, _ = model(t)
            p = torch.sigmoid(sp).cpu().numpy().squeeze()
        p = np.atleast_1d(p).astype(np.float32)
        preds[start:end]  += p[:end-start]
        counts[start:end] += 1
        del t, sp, _
        torch.cuda.empty_cache()
        if end >= n: break
        start = end - ovl
    mask = counts > 0
    preds[mask] /= counts[mask]
    return preds

# ──── Scene Boundary ─────────────────────────────────────────────────────────────
def pick_cuts(prob: np.ndarray, thr=CUT_THR, min_d=MIN_CUT_DIST) -> list[int]:
    p, N = np.asarray(prob).ravel(), len(prob)
    mask, cuts = p >= thr, []
    i = 0
    while i < N:
        if mask[i]:
            j = i
            while j + 1 < N and mask[j+1]: j += 1
            cuts.append(i + int(np.argmax(p[i:j+1])))
            i = j + 1
        else:
            i += 1
    pruned = []
    for c in cuts:
        if not pruned or c - pruned[-1] >= min_d: pruned.append(c)
        elif p[c] > p[pruned[-1]]:               pruned[-1] = c
    return pruned

def cuts_to_scenes(cuts: list[int], n: int) -> list[tuple[int,int]]:
    cuts   = sorted(int(c) for c in cuts if 0 <= c < n)
    starts = [0]    + [c+1 for c in cuts]
    ends   = cuts   + [n-1]
    return [(s,e) for s,e in zip(starts,ends) if e-s+1 >= MIN_SCENE_LEN]

def sample_scene(s: int, e: int, n: int = N_SAMPLE_PER_SCENE) -> list[int]:
    if e - s + 1 <= n: return list(range(s, e+1))
    return np.linspace(s, e, n, dtype=int).tolist()

# ──── Alg.1 Frame Filtering (Paper §3.2.3, cos > 0.9 → near-duplicate) ──────────
@torch.no_grad()
def alg1_dedup_filter(frames_48x27: np.ndarray,
                      selected_per_scene: list[list[int]],
                      model, preprocess,
                      S: float = DEDUP_THR) -> list[list[int]]:
    all_idx = list({i for scene in selected_per_scene for i in scene})
    if not all_idx: return [[] for _ in selected_per_scene]
    emb_map: dict[int, np.ndarray] = {}
    for start in range(0, len(all_idx), 64):
        chunk_idx = all_idx[start:start+64]
        pils = [Image.fromarray(frames_48x27[i]) for i in chunk_idx]
        t    = torch.stack([preprocess(p) for p in pils]).to(DEVICE)
        f    = model.encode_image(t)
        f    = (f / f.norm(dim=-1, keepdim=True)).cpu().numpy()
        for ci, feat in zip(chunk_idx, f):
            emb_map[ci] = feat
    def cos(a, b): return float(np.dot(a, b))
    kept_all = []
    for scene in selected_per_scene:
        if not scene: kept_all.append([]); continue
        kept = [scene[0]]
        for t in range(1, len(scene)):
            e_t  = emb_map[scene[t]]
            drop = any(cos(e_t, emb_map[k]) >= S for k in kept)
            if not drop: kept.append(scene[t])
        kept_all.append(kept)
    return kept_all

# ──── Save Keyframes (cv2 seek → ffmpeg fallback cho H.265/HEVC) ──────────────────
def save_keyframes(video_path: str, kept_per_scene: list[list[int]],
                   video_name: str, fps: float, out_dir: str,
                   quality: int = JPEG_QUALITY) -> list[dict]:
    import subprocess as _sp2
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    target_map: dict[int, tuple[int,int]] = {}
    for sid, scene in enumerate(kept_per_scene):
        for rank, fi in enumerate(scene):
            target_map[fi] = (sid, rank)
    if not target_map: return []
    sorted_targets = sorted(target_map.keys())

    # Method 1: cv2 seek (fast, H.264 CFR)
    local_tmp = f"/tmp/_aic_tmp_{video_name}.mp4"
    if "/content/drive" in video_path and not os.path.exists(local_tmp):
        shutil.copy2(video_path, local_tmp)
    read_path = local_tmp if os.path.exists(local_tmp) else video_path
    frame_buf: dict[int, np.ndarray] = {}
    cap = cv2.VideoCapture(read_path)
    if cap.isOpened():
        n_cv2 = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for fi in sorted_targets:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frm = cap.read()
            if ret: frame_buf[fi] = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB)
        cap.release()
        if not frame_buf:
            print(f"  [WARN] cv2 seek 0/{len(sorted_targets)} frames "
                  f"(cv2_total={n_cv2}, max_target={sorted_targets[-1]}) "
                  f"→ H.265/HEVC detected, switching to ffmpeg select fallback")

    # Method 2: ffmpeg select (reliable, works H.265/HEVC)
    if not frame_buf:
        ffmp_tmp = f"/tmp/_aic_ffout_{video_name}"
        Path(ffmp_tmp).mkdir(exist_ok=True)
        try:
            sel = "+".join(f"eq(n\\,{fi})" for fi in sorted_targets)
            cmd = ["ffmpeg", "-y", "-hwaccel", "auto", "-i", read_path,
                   "-vf", f"select='{sel}'", "-vsync", "vfr", "-q:v", "2",
                   os.path.join(ffmp_tmp, "f_%06d.jpg")]
            _sp2.run(cmd, capture_output=True, timeout=900)
            extracted = sorted(glob.glob(os.path.join(ffmp_tmp, "f_*.jpg")))
            for fi, fpath in zip(sorted_targets, extracted):
                if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
                    frame_buf[fi] = np.array(Image.open(fpath).convert("RGB"))
            print(f"  [FFmpeg] Extracted {len(frame_buf)}/{len(sorted_targets)} frames")
        except Exception as e:
            print(f"  [ERROR] ffmpeg fallback failed: {e}")
        finally:
            shutil.rmtree(ffmp_tmp, ignore_errors=True)

    if os.path.exists(local_tmp): os.remove(local_tmp)
    if not frame_buf:
        print(f"  [ERROR] Both methods failed for {video_name}"); return []

    first = next(iter(frame_buf.values()))
    H_img, W_img = first.shape[:2]
    meta_list = []
    for fi, (sid, rank) in sorted(target_map.items(), key=lambda x: (x[1][0], x[1][1])):
        if fi not in frame_buf: continue
        fname    = f"{video_name}-{sid:04d}-{fi}.jpg"
        ts_ms    = int(fi / fps * 1000)
        ts_str   = (f"{ts_ms//3600000:02d}:{(ts_ms%3600000)//60000:02d}:"
                    f"{(ts_ms%60000)//1000:02d}.{ts_ms%1000:03d}")
        fpath    = os.path.join(out_dir, fname)
        Image.fromarray(frame_buf[fi]).save(fpath, "JPEG", quality=quality)
        meta_list.append({
            "frame":         f"/static/images/{fname}",
            "url":           f"http://localhost:8000/static/images/{fname}",
            "name":          fname,
            "video":         video_name,
            "scene_id":      sid,
            "frame_idx":     fi,
            "fps":           fps,
            "width":         W_img,
            "height":        H_img,
            "timestamp_ms":  ts_ms,
            "timestamp_str": ts_str,
            "content": [], "ocr": [], "object": [], "color": [], "action": [],
            "faiss_id_clip":   -1,
            "faiss_id_beit3":  -1,
            "neighbors_clip":  [],
        })
    return meta_list

# ──── ZIP & Upload to Drive ───────────────────────────────────────────────────────
def zip_and_upload(video_names: list[str], zip_id: int,
                   local_kf_dir: str, zip_dir: str) -> str:
    zip_name = f"AIC2026_frames_{zip_id:03d}.zip"
    zip_path = os.path.join(zip_dir, zip_name)
    print(f"  [Zip] Creating {zip_name} ({len(video_names)} videos)...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for vn in video_names:
            vdir = os.path.join(local_kf_dir, vn)
            if not os.path.isdir(vdir): continue
            for f in os.listdir(vdir):
                zf.write(os.path.join(vdir, f), arcname=f"{vn}/{f}")
    sz_mb = os.path.getsize(zip_path) / 1024**2
    print(f"  [Zip] {zip_name}: {sz_mb:.0f} MB in {time.time()-t0:.0f}s")
    for vn in video_names:
        vdir = os.path.join(local_kf_dir, vn)
        if os.path.isdir(vdir): shutil.rmtree(vdir)
    return zip_name

print("[SECTION 06] Helper functions OK")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 06b – CÁCH CẮT 2: LẤY MẪU THEO TỐC ĐỘ  (ghi đè hàm của SECTION 06)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
# Khác cách 1 ở ba điểm:
#   1. Số ảnh phụ thuộc ĐỘ DÀI cảnh, không cố định 4. Cảnh 8 frame và cảnh 4500
#      frame ở cách 1 đều được đúng 4 ảnh — cảnh dài bị bỏ sót nội dung.
#   2. fps lấy từ BẢNG CỦA BTC, không dò bằng ffprobe. BTC dùng chính bảng này
#      để đánh số frame đáp án; ffprobe có thể trả 30.0 ở video BTC ghi 29.97 —
#      lệch 20%, tới phút 20 là sai gần 4 phút, đủ trượt hẳn cảnh chứa đáp án.
#   3. KHÔNG lọc trùng ở bước cắt. Paper §3.2.3 lọc SAU khi trích đặc trưng,
#      dùng chính embedding nạp FAISS. Lọc trên buffer 48×27 thì chữ trên màn
#      hình đã biến mất trước khi so, nên nó xoá ngẫu nhiên giữa frame còn chữ
#      và frame đã đổi chữ.

FLOOR_SECONDS   = 50 / 30    # ≈ 1,67 giây — hết sàn 2 ảnh
SECONDS_PER_EXTRA = 2.0        # sau sàn, cứ 2 giây thêm 1 ảnh
N_MIN, N_MAX   = 2, 40
MARGIN         = 3          # lùi 3 frame mỗi đầu, tránh frame chuyển cảnh

# Bảng fps của BTC. Đẩy data_raw/fps_map.json trong repo lên đường dẫn này.
FPS_MAP_PATH = f"{SHARED_DIR}/fps_map.json"
if not os.path.exists(FPS_MAP_PATH):
    raise FileNotFoundError(
        f"Thiếu {FPS_MAP_PATH}\n"
        "  → Chép data_raw/fps_map.json trong repo lên đúng đường dẫn đó.\n"
        "  Cách 2 KHÔNG dò fps từ video: đây là số chốt của BTC."
    )
with open(FPS_MAP_PATH) as f:
    FPS_MAP: dict[str, float] = {k: float(v) for k, v in json.load(f).items()}

# Counter dùng lại ở SECTION 07 bản cách 2 để in phân bố số ảnh mỗi cảnh.
# SECTION 02 chỉ import defaultdict nên phải thêm ở đây.
from collections import Counter

# Cách 2 bỏ SECTION 05 (nạp CLIP ViT-B/32 để lọc trùng) — bỏ luôn cả dòng
# `import open_clip` nằm trong đó, mà SECTION 10 cần khi dựng index CLIP.
# Gói open_clip_torch đã được cài ở SECTION 00 nên chỉ thiếu mỗi dòng import.
import open_clip
print(f"[fps] {len(FPS_MAP)} video · " +
      " · ".join(f"{k}fps: {v}" for k, v in sorted(Counter(FPS_MAP.values()).items())))

# Video không có trong bảng thì không cắt được — loại khỏi hàng đợi luôn.
_no_fps = [v for v in todo_list if Path(v).stem not in FPS_MAP]
todo_list  = [v for v in todo_list if Path(v).stem in FPS_MAP]
if _no_fps:
    print(f"[Skip]  {len(_no_fps)} videos not in the fps table: "
          f"{[Path(v).stem for v in _no_fps[:5]]}"
          f"{' …' if len(_no_fps) > 5 else ''}")
    print(f"[Left] {len(todo_list)} videos left to cut")


def get_video_fps(path: str):
    """Ghi đè hàm SECTION 06. None nếu video không có trong bảng BTC."""
    return FPS_MAP.get(Path(path).stem)


def sample_scene(s: int, e: int, fps: float = 25.0) -> list[int]:
    """Số ảnh theo độ dài cảnh.

    n = 2 cho tới 1,67 giây; sau đó cứ thêm 2 giây thì thêm 1 ảnh; kẹp [2, 40].
    Quy ra frame @30fps:  8–50 → 2 ảnh · 51–110 → 3 · 111–170 → 4
                 @25fps:  8–42 → 2 ảnh · 43–92  → 3 · 93–142  → 4
    Mốc theo GIÂY giống nhau ở mọi fps, nên video 25 và 30 fps có cùng mật độ
    ảnh trên mỗi giây.
    """
    L = e - s + 1
    _floor_frames  = max(1, int(round(fps * FLOOR_SECONDS)))
    _step_frames = max(1, int(round(fps * SECONDS_PER_EXTRA)))
    n = N_MIN if L <= _floor_frames else N_MIN + -(-(L - _floor_frames) // _step_frames)
    n = max(N_MIN, min(N_MAX, n))
    if L <= n:
        return list(range(s, e + 1))
    m = min(MARGIN, (L - n) // 2)
    return np.linspace(s + m, e - m, n, dtype=int).tolist()


print("[Method2] Overrode sample_scene() and get_video_fps().")
print(f"[Method2] n=2 up to {FLOOR_SECONDS:.2f}s, then +1 frame every {SECONDS_PER_EXTRA}s, "
      f"clamp [{N_MIN}, {N_MAX}], inset {MARGIN} frames.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 07 – Vòng lặp cắt · CÁCH 2 (auto-resume + auto-zip)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

if not todo_list:
    print("[SKIP] All videos already cut. Go to Section 09.")
else:
    BUFFER_META: list[dict] = []
    # Cách 2: FPS_MAP là bảng BTC nạp ở SECTION 06b — KHÔNG đè.
    # FPS_USED chỉ để ghi lại fps của các video đã cắt, phục vụ bước index.
    FPS_USED:     dict[str, float] = {}
    fps_map_path = os.path.join(INDEX_DIR, "fps_map.json")

    print(f"\n[Main Loop] {len(todo_list)} videos | Buffer trigger: {ZIP_EVERY_GB} GB\n")

    for video_path in tqdm(todo_list, desc="Cutting"):
        video_name = Path(video_path).stem
        out_dir    = os.path.join(LOCAL_KF_DIR, video_name)
        t0         = time.time()
        print(f"\n  [{video_name}] START")

        fps = get_video_fps(video_path)
        FPS_USED[video_name] = fps

        try:
            frames_tnv2 = decode_video_tnv2(video_path)
        except Exception as e:
            print(f"  [ERROR] decode {video_name}: {e}"); continue

        n_frames = frames_tnv2.shape[0]
        print(f"  [1/4] Decoded 48×27: {n_frames} frames ({n_frames/fps:.0f}s)")

        prob   = run_tnv2_chunked(frames_tnv2, tnv2)
        cuts   = pick_cuts(prob)
        scenes = cuts_to_scenes(cuts, n_frames)
        print(f"  [2/4] TransNetV2: {len(cuts)} cuts → {len(scenes)} scenes")

        # Cách 2: số ảnh theo độ dài cảnh, và KHÔNG lọc trùng ở đây.
        sampled = [sample_scene(s, e, fps) for s, e in scenes]
        kept    = sampled
        _total   = sum(len(x) for x in kept)
        _dist   = Counter(len(x) for x in kept)
        print(f"  [3/4] Rate-based sampling: {_total} frames · "
              + " · ".join(f"{k}f x{v}" for k, v in sorted(_dist.items())[:6]))
        del frames_tnv2; torch.cuda.empty_cache()

        meta = save_keyframes(video_path, kept, video_name, fps, out_dir)
        print(f"  [4/4] Saved {len(meta)} keyframes → {out_dir}")

        PROGRESS["done_videos"].append(video_name)
        PROGRESS["buffer_videos"].append(video_name)
        PROGRESS["total_frames_cut"] += len(meta)
        buf_size = sum(
            os.path.getsize(os.path.join(out_dir, f))
            for f in os.listdir(out_dir) if os.path.isfile(os.path.join(out_dir, f))
        ) if os.path.isdir(out_dir) else 0
        PROGRESS["buffer_size_bytes"] += buf_size
        BUFFER_META.extend(meta)
        save_progress(PROGRESS)
        with open(fps_map_path, "w") as f: json.dump(FPS_USED, f, indent=2)
        print(f"  [{video_name}] DONE in {time.time()-t0:.0f}s | "
              f"Buffer: {PROGRESS['buffer_size_bytes']/1024**3:.2f} GB")

        if PROGRESS["buffer_size_bytes"] >= ZIP_EVERY_GB * 1024**3:
            zid      = PROGRESS["zip_count"] + 1
            zip_name = zip_and_upload(PROGRESS["buffer_videos"], zid, LOCAL_KF_DIR, ZIP_DIR)
            PROGRESS["zip_count"]         = zid
            PROGRESS["completed_zips"].append(zip_name)
            PROGRESS["buffer_videos"]     = []
            PROGRESS["buffer_size_bytes"] = 0
            BUFFER_META = []
            save_progress(PROGRESS)
            print(f"  [Zip] ✓ {zip_name}")

    if PROGRESS["buffer_videos"]:
        zid      = PROGRESS["zip_count"] + 1
        zip_name = zip_and_upload(PROGRESS["buffer_videos"], zid, LOCAL_KF_DIR, ZIP_DIR)
        PROGRESS["zip_count"]         = zid
        PROGRESS["completed_zips"].append(zip_name)
        PROGRESS["buffer_videos"]     = []
        PROGRESS["buffer_size_bytes"] = 0
        save_progress(PROGRESS)
        print(f"[Final Zip] ✓ {zip_name}")

    print(f"\n[✓] Cutting done! Total: {PROGRESS['total_frames_cut']:,} keyframes, "
          f"{PROGRESS['zip_count']} zips")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 08 – Status Check (run bất kỳ lúc nào)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
zip_gb = sum(os.path.getsize(os.path.join(ZIP_DIR, z)) / 1024**3
             for z in p["completed_zips"]
             if os.path.exists(os.path.join(ZIP_DIR, z)))
print("=" * 60)
print(f"  PIPELINE STATUS  [{MODE} mode]")
print("=" * 60)
print(f"  Videos done        : {len(p['done_videos'])} / {len(ALL_VIDEOS)}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']} ({zip_gb:.1f} GB)")
print(f"  BEiT3 indexed      : {len(p.get('index_done_beit3', []))} / {p['zip_count']}")
print(f"  CLIP indexed       : {len(p['index_done_clip'])} / {p['zip_count']}")
print(f"  Index done         : {p['index_phase_done']}")
print(f"  OCR done           : {len(p.get('done_ocr', []))} videos")
print(f"  ASR done           : {len(p.get('done_asr', []))} videos")
print(f"  Caption done       : {len(p.get('done_caption', []))} videos")
print(f"  Features merged    : {p.get('features_merged', False)}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 09 – Build BEiT3 Index (stream through zips, resumable)
# Architecture: BEiT3-large (Wang et al., CVPR 2023) — Multiway Transformer,
#   675M params, backbone qua package torchscale (Magneto architecture).
#   Checkpoint: beit3_large_patch16_384_coco_retrieval.pth (fine-tuned COCO
#   retrieval, dual-encoder ITC). Đây CHÍNH XÁC là model [51] paper 2504.08384
#   cite cho nhánh "fine-grained" trong Ensemble Search.
# Toàn bộ setup dưới đây đã tự verify bằng cách clone source thật từ
# microsoft/unilm, cài dependency, patch lỗi, tải checkpoint, chạy forward
# pass thành công trong sandbox trước khi đưa vào đây — KHÔNG đoán code.
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import faiss

# ── Bước 1: Clone source code thật (không phải pip package chuẩn) ─────────────
if not os.path.exists(BEIT3_CODE_DIR):
    print("[BEiT3] Cloning microsoft/unilm (sparse, beit3/ only)...")
    _sp.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "https://github.com/microsoft/unilm.git", BEIT3_CODE_DIR],
           check=True, capture_output=True)
    _sp.run(["git", "-C", BEIT3_CODE_DIR, "sparse-checkout", "set", "beit3"],
           check=True, capture_output=True)
    print(f"[BEiT3] Cloned → {BEIT3_CODE_DIR}/beit3")

BEIT3_SRC = os.path.join(BEIT3_CODE_DIR, "beit3")

# ── Bước 2: Patch torch._six (verify: module đã bị xóa khỏi torch hiện đại,
#    unilm/beit3/utils.py dòng 24 dùng "from torch._six import inf" → crash).
#    Patch 1 dòng, well-known standard fix, không đổi logic gì khác. ─────────
_utils_py = os.path.join(BEIT3_SRC, "utils.py")
with open(_utils_py) as f: _utils_content = f.read()
if "from torch._six import inf" in _utils_content:
    _utils_content = _utils_content.replace(
        "from torch._six import inf",
        "inf = float('inf')  # torch._six removed in modern torch — standard patch"
    )
    with open(_utils_py, "w") as f: f.write(_utils_content)
    print("[BEiT3] Patched torch._six issue trong utils.py")

if BEIT3_SRC not in sys.path:
    sys.path.insert(0, BEIT3_SRC)

# ── Bước 3: Tải checkpoint + tokenizer thật (verify link tải OK, 1.29GB + 1.3MB) ─
_beit3_ckpt_path = os.path.join(OUTPUT_ROOT, "beit3_large_patch16_384_coco_retrieval.pth")
_beit3_spm_path  = os.path.join(OUTPUT_ROOT, "beit3.spm")
if not os.path.exists(_beit3_ckpt_path):
    print("[BEiT3] Downloading checkpoint (1.29GB)...")
    _sp.run(["curl", "-sL", "-o", _beit3_ckpt_path, BEIT3_CKPT_URL], check=True)
if not os.path.exists(_beit3_spm_path):
    print("[BEiT3] Downloading tokenizer...")
    _sp.run(["curl", "-sL", "-o", _beit3_spm_path, BEIT3_SPM_URL], check=True)

p = load_progress()
remaining_beit3 = [z for z in p["completed_zips"] if z not in p.get("index_done_beit3", [])]
p.setdefault("index_done_beit3", [])

if not remaining_beit3:
    print("[SKIP] BEiT3 index already up to date"
         f"{' (index_phase_done=True từ lần chạy trước)' if p['index_phase_done'] else ''}.")
else:
    if p["index_phase_done"]:
        print(f"[Resume] Found {len(remaining_beit3)} new zips not yet indexed "
             f"kể từ lần index_phase_done trước → tiếp tục index.")

    from timm.models import create_model
    import modeling_finetune  # trigger @register_model registration
    from transformers import XLMRobertaTokenizer
    import torchvision.transforms as T

    print(f"[BEiT3] Loading {BEIT3_MODEL_NAME}...")
    beit3_model = create_model(BEIT3_MODEL_NAME, pretrained=False)
    _ckpt = torch.load(_beit3_ckpt_path, map_location="cpu")
    _sd   = _ckpt["model"] if "model" in _ckpt else _ckpt
    del _ckpt; gc.collect() if 'gc' in dir() else None
    _missing, _unexpected = beit3_model.load_state_dict(_sd, strict=False)
    del _sd
    print(f"[BEiT3] load_state_dict: missing={len(_missing)}, unexpected={len(_unexpected)}")
    beit3_model.eval().to(DEVICE)
    beit3_tokenizer = XLMRobertaTokenizer(_beit3_spm_path)
    print(f"[BEiT3] Loaded. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f}GB")

    # Image transform — verify chính xác từ unilm/beit3/datasets.py build_transform()
    # (eval path, is_train=False): Resize BICUBIC → ToTensor → Normalize Inception stats
    _beit3_transform = T.Compose([
        T.Resize((BEIT3_INPUT_SIZE, BEIT3_INPUT_SIZE), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),  # IMAGENET_INCEPTION stats, verify từ timm.data.constants
    ])

    beit3_idx_path  = os.path.join(INDEX_DIR, "beit3.index")
    beit3_meta_path = os.path.join(INDEX_DIR, "beit3_mapping.json")
    if os.path.exists(beit3_idx_path):
        beit3_index = faiss.read_index(beit3_idx_path)
        with open(beit3_meta_path) as f: beit3_map = json.load(f)
        print(f"[BEiT3] Loaded existing: {beit3_index.ntotal} vectors")
    else:
        beit3_index = faiss.IndexFlatIP(FAISS_BEIT3_DIM)
        beit3_map   = {}

    full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
    if os.path.exists(full_meta_path):
        with open(full_meta_path) as f: FULL_META = json.load(f)
        fname_to_idx = {m["name"]: i for i, m in enumerate(FULL_META)}
    else:
        FULL_META = []; fname_to_idx = {}

    UNZIP_DIR = "/content/unzip_buf"

    @torch.no_grad()
    def embed_beit3(img_paths: list[str], bs: int = BEIT3_BATCH) -> np.ndarray:
        """API verify thật từ BEiT3ForRetrieval.forward(): image=X, only_infer=True
        → (vision_cls, None). vision_cls đã qua F.normalize() sẵn trong model."""
        all_feats = []
        for s in range(0, len(img_paths), bs):
            chunk = img_paths[s:s+bs]
            imgs  = [Image.open(p).convert("RGB") for p in chunk]
            t     = torch.stack([_beit3_transform(img) for img in imgs]).to(DEVICE)
            vision_cls, _ = beit3_model(image=t, only_infer=True)
            all_feats.append(vision_cls.cpu().numpy())
        embeds = np.vstack(all_feats).astype("float32")
        if embeds.shape[1] != FAISS_BEIT3_DIM:
            raise ValueError(
                f"[BEiT3] Model output dim={embeds.shape[1]} KHÁC "
                f"FAISS_BEIT3_DIM={FAISS_BEIT3_DIM} đã khai báo trong config. "
                f"Sửa FAISS_BEIT3_DIM = {embeds.shape[1]} rồi chạy lại."
            )
        return embeds

    for zip_name in tqdm(remaining_beit3, desc="BEiT3 indexing"):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): print(f"  [WARN] {zip_name} not found"); continue
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        Path(UNZIP_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  [Unzip] {zip_name}...")
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(UNZIP_DIR)
        img_paths = sorted(glob.glob(os.path.join(UNZIP_DIR, "**", "*.jpg"), recursive=True))
        if not img_paths: shutil.rmtree(UNZIP_DIR, ignore_errors=True); continue
        print(f"  [BEiT3] Embedding {len(img_paths)} images...")
        embeds   = embed_beit3(img_paths)
        start_id = beit3_index.ntotal
        beit3_index.add(embeds)
        for i, path in enumerate(img_paths):
            fname    = os.path.basename(path)
            faiss_id = start_id + i
            beit3_map[str(faiss_id)] = f"http://localhost:8000/static/images/{fname}"
            if fname in fname_to_idx:
                FULL_META[fname_to_idx[fname]]["faiss_id_beit3"] = faiss_id
            else:
                parts    = fname.rsplit("-", 2)
                video    = parts[0]
                scene_id = int(parts[1]) if len(parts) > 2 else 0
                fi       = int(parts[2].replace(".jpg","")) if len(parts) > 2 else 0
                fps_v    = FPS_MAP.get(video, 25.0) if 'FPS_MAP' in dir() else 25.0
                ts_ms    = int(fi / fps_v * 1000)
                ts_str   = (f"{ts_ms//3600000:02d}:{(ts_ms%3600000)//60000:02d}:"
                            f"{(ts_ms%60000)//1000:02d}.{ts_ms%1000:03d}")
                entry = {
                    "faiss_id_beit3": faiss_id, "faiss_id_clip": -1,
                    "frame": f"/static/images/{fname}",
                    "url":   f"http://localhost:8000/static/images/{fname}",
                    "name":  fname, "video": video, "scene_id": scene_id,
                    "frame_idx": fi, "fps": fps_v, "timestamp_ms": ts_ms,
                    "timestamp_str": ts_str,
                    "content": [], "ocr": [], "object": [], "color": [], "action": [],
                    "neighbors_clip": [],
                }
                fname_to_idx[fname] = len(FULL_META)
                FULL_META.append(entry)
        faiss.write_index(beit3_index, beit3_idx_path)
        with open(beit3_meta_path, "w") as f: json.dump(beit3_map, f)
        with open(full_meta_path, "w") as f:   json.dump(FULL_META, f, indent=2)
        p["index_done_beit3"].append(zip_name)
        save_progress(p)
        print(f"  [✓] {zip_name}: +{len(img_paths)} → total {beit3_index.ntotal}")
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        torch.cuda.empty_cache()
    print(f"[BEiT3] Done! Total: {beit3_index.ntotal}")
    del beit3_model; torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 10 – Build CLIP ViT-H-14 Index
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
remaining_clip = [z for z in p["completed_zips"] if z not in p["index_done_clip"]]

# FIX: giống BEiT3 ở trên — check remaining_clip trước, không gate theo
# index_phase_done (flag cũ có thể stale khi chuyển TEST→FULL mode).
if not remaining_clip:
    print("[SKIP] CLIP index already up to date"
         f"{' (index_phase_done=True từ lần chạy trước)' if p['index_phase_done'] else ''}.")
else:
    if p["index_phase_done"]:
        print(f"[Resume] Found {len(remaining_clip)} new zips not yet indexed "
             f"kể từ lần index_phase_done trước → tiếp tục index.")
    print(f"[CLIP] Loading {CLIP_MODEL} / {CLIP_PRETRAIN}...")
    clip_model_idx, _, clip_prep_idx = open_clip.create_model_and_transforms(
        CLIP_MODEL, pretrained=CLIP_PRETRAIN, device=DEVICE
    )
    clip_model_idx.eval()
    if 'clip_filter_model' in dir() and hasattr(clip_filter_model, 'parameters'):
        try: del clip_filter_model; torch.cuda.empty_cache()
        except: pass

    clip_idx_path  = os.path.join(INDEX_DIR, "clip.index")
    clip_meta_path = os.path.join(INDEX_DIR, "clip_mapping.json")
    if os.path.exists(clip_idx_path):
        clip_index = faiss.read_index(clip_idx_path)
        with open(clip_meta_path) as f: clip_map = json.load(f)
        print(f"[CLIP] Loaded existing: {clip_index.ntotal} vectors")
    else:
        clip_index = faiss.IndexFlatIP(FAISS_CLIP_DIM)
        clip_map   = {}

    full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
    if not ('FULL_META' in dir() and FULL_META):
        with open(full_meta_path) as f: FULL_META = json.load(f)
    fname_to_idx = {m["name"]: i for i, m in enumerate(FULL_META)}

    UNZIP_DIR = "/content/unzip_buf"

    @torch.no_grad()
    def embed_images_clip(img_paths: list[str], bs: int = CLIP_BATCH) -> np.ndarray:
        all_feats = []
        for s in range(0, len(img_paths), bs):
            chunk = img_paths[s:s+bs]
            pils  = [Image.open(p).convert("RGB") for p in chunk]
            t     = torch.stack([clip_prep_idx(img) for img in pils]).to(DEVICE)
            f     = clip_model_idx.encode_image(t)
            f     = f / f.norm(dim=-1, keepdim=True)
            all_feats.append(f.cpu().numpy())
        embeds = np.vstack(all_feats).astype("float32")
        # Cùng safety net như BEiT3 — tránh lặp lại bug hardcode-sai-dim khi
        # đổi model (vừa đổi ViT-H-14→ViT-bigG-14, dim 1024→1280).
        if embeds.shape[1] != FAISS_CLIP_DIM:
            raise ValueError(
                f"[CLIP] Model output dim={embeds.shape[1]} KHÁC "
                f"FAISS_CLIP_DIM={FAISS_CLIP_DIM} đã khai báo trong config. "
                f"Sửa FAISS_CLIP_DIM = {embeds.shape[1]} rồi chạy lại."
            )
        return embeds

    for zip_name in tqdm(remaining_clip, desc="CLIP indexing"):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): print(f"  [WARN] {zip_name} not found"); continue
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        Path(UNZIP_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  [Unzip] {zip_name}...")
        with zipfile.ZipFile(zip_path) as zf: zf.extractall(UNZIP_DIR)
        img_paths = sorted(glob.glob(os.path.join(UNZIP_DIR, "**", "*.jpg"), recursive=True))
        if not img_paths: shutil.rmtree(UNZIP_DIR, ignore_errors=True); continue
        print(f"  [CLIP] Embedding {len(img_paths)} images...")
        embeds   = embed_images_clip(img_paths)
        start_id = clip_index.ntotal
        clip_index.add(embeds)
        for i, path in enumerate(img_paths):
            fname    = os.path.basename(path)
            faiss_id = start_id + i
            clip_map[str(faiss_id)] = f"http://localhost:8000/static/images/{fname}"
            if fname in fname_to_idx:
                FULL_META[fname_to_idx[fname]]["faiss_id_clip"] = faiss_id
        faiss.write_index(clip_index, clip_idx_path)
        with open(clip_meta_path, "w") as f: json.dump(clip_map, f)
        with open(full_meta_path, "w") as f:  json.dump(FULL_META, f, indent=2)
        p["index_done_clip"].append(zip_name)
        save_progress(p)
        print(f"  [✓] {zip_name}: +{len(img_paths)} → total {clip_index.ntotal}")
        shutil.rmtree(UNZIP_DIR, ignore_errors=True)
        torch.cuda.empty_cache()
    print(f"[CLIP] Done! Total: {clip_index.ntotal}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 11 – Neighbor Lists + All Mapping Files
# ═══════════════════════════════════════════════════════════════════════════════
# %%

full_meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
with open(full_meta_path) as f: FULL_META = json.load(f)
print(f"[Meta] Loaded {len(FULL_META)} keyframe entries")

video_groups: dict[str, list[int]] = defaultdict(list)
for i, m in enumerate(FULL_META):
    video_groups[m["video"]].append(i)
for video, indices in video_groups.items():
    video_groups[video] = sorted(indices, key=lambda i: FULL_META[i]["frame_idx"])

for video, indices in video_groups.items():
    n = len(indices)
    for pos, meta_idx in enumerate(indices):
        nbr_clip_ids = []
        for offset in range(-NEIGHBOR_WINDOW, NEIGHBOR_WINDOW + 1):
            if offset == 0: continue
            nbr_pos = pos + offset
            if 0 <= nbr_pos < n:
                nbr_meta = FULL_META[indices[nbr_pos]]
                cid = nbr_meta.get("faiss_id_clip", -1)
                if cid >= 0: nbr_clip_ids.append(cid)
        FULL_META[meta_idx]["neighbors_clip"] = nbr_clip_ids

with open(full_meta_path, "w") as f: json.dump(FULL_META, f, indent=2)
print(f"[Meta] Saved with neighbor lists")

# keyframes_list.json
kf_list = [f"{m['video']}-{m['frame_idx']}" for m in FULL_META]
with open(os.path.join(INDEX_DIR, "keyframes_list.json"), "w") as f:
    json.dump(sorted(set(kf_list)), f, indent=2)
print(f"[Meta] keyframes_list.json: {len(set(kf_list))} entries")

# image_mapping.json (backward compat: CLIP faiss_id → url)
image_mapping = {str(m.get("faiss_id_clip",-1)): m["url"]
                 for m in FULL_META if m.get("faiss_id_clip",-1) >= 0}
with open(os.path.join(INDEX_DIR, "image_mapping.json"), "w") as f:
    json.dump(image_mapping, f, indent=2)
print(f"[Meta] image_mapping.json: {len(image_mapping)} entries")

# fps_map.json
fps_unique = {m["video"]: m["fps"] for m in FULL_META}
with open(os.path.join(INDEX_DIR, "fps_map.json"), "w") as f:
    json.dump(fps_unique, f, indent=2)
print(f"[Meta] fps_map.json: {len(fps_unique)} videos")

p = load_progress(); p["index_phase_done"] = True; save_progress(p)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 12 – Summary Part 1 (Baseline Complete)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

p = load_progress()
print("=" * 60)
print(f"  PART 1 COMPLETE  [{MODE} mode]")
print("=" * 60)
print(f"  Videos cut         : {len(p['done_videos'])}")
print(f"  Total keyframes    : {p['total_frames_cut']:,}")
print(f"  Zips on Drive      : {p['zip_count']}")
print(f"  Index phase done   : {p['index_phase_done']}")
print()
for fname, desc in {
    "beit3.index":            "BEiT3-large FAISS (1024-dim, fine-grained)",
    "clip.index":             "CLIP ViT-bigG-14 FAISS (1280-dim, coarse-grained)",
    "keyframe_metadata.json": "Rich metadata",
    "keyframes_list.json":    "UI submit form",
    "image_mapping.json":     "Backward compat",
    "fps_map.json":           "FPS per video",
}.items():
    fpath = os.path.join(INDEX_DIR, fname)
    if os.path.exists(fpath):
        print(f"  ✓ {fname:<30} {os.path.getsize(fpath)/1024**2:7.1f} MB  ({desc})")
    else:
        print(f"  ✗ {fname:<30}  MISSING")
print()
print("  -> Next: run nb_full_part2_features.py to extract OCR/ASR/Caption/Tags/Objects/Color")

---

## Phòng lab — đánh giá model trên frame vừa cắt

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Unzip 2000 frames (stratified từ mọi video)
# ═══════════════════════════════════════════════════════════════════════════════
# %%

with open(f"{OUTPUT_ROOT}/progress.json") as f: prog = json.load(f)
zips = prog.get("completed_zips", [])
assert zips, "Chưa có zip. Chạy Part 1 trước."

meta_path = os.path.join(INDEX_DIR, "keyframe_metadata.json")
META_BY_NAME = {}
if os.path.exists(meta_path):
    with open(meta_path) as f: full_meta = json.load(f)
    META_BY_NAME = {m["name"]: m for m in full_meta}

per_zip = max(1, N_EVAL_FRAMES // len(zips))
extracted = []
for zip_name in zips:
    zip_path = os.path.join(ZIP_DIR, zip_name)
    if not os.path.exists(zip_path): continue
    with zipfile.ZipFile(zip_path) as zf:
        imgs = [n for n in zf.namelist() if n.endswith(".jpg")]
        by_video = defaultdict(list)
        for n in imgs: by_video[n.split("/")[0]].append(n)
        per_v    = max(1, per_zip // max(len(by_video), 1))
        selected = []
        for v_imgs in sorted(by_video.values()):
            step = max(1, len(v_imgs) // per_v)
            selected.extend(v_imgs[::step][:per_v])
        selected = selected[:per_zip]
        for member in selected:
            dest = os.path.join(EVAL_DIR, os.path.basename(member))
            with zf.open(member) as src, open(dest, "wb") as dst:
                dst.write(src.read())
        extracted.extend([os.path.join(EVAL_DIR, os.path.basename(m)) for m in selected])

EVAL_FRAMES = sorted(set(f for f in extracted if os.path.exists(f)))
print(f"[Eval] {len(EVAL_FRAMES)} frames extracted from {len(zips)} zips")
# Video coverage
vid_set = {os.path.basename(f).split("-")[0] for f in EVAL_FRAMES}
print(f"[Eval] Covers {len(vid_set)} unique videos")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03 – OCR Region Validation  (PaddleOCR detect + VietOCR recognize)
# Architecture: PaddleOCR DBNet++ (detect ONLY) → VietOCR vgg_transformer (recognize)
#   Two-stage, KHÔNG dùng PaddleOCR built-in lang="vi" recognizer.
# Root cause của garbled text (chữ lẫn ö/ü/á thay vì đúng dấu tiếng Việt):
#   PaddleOCR lang="vi" dùng 1 recognition model đa ngôn ngữ chung (Latin family),
#   KHÔNG train riêng cho dấu tiếng Việt → hay đọc nhầm dấu.
# Grounded fix — bám sát ĐÚNG baseline gốc của team (running.py, OCR2json()):
#   detector = PaddleOCR(...); detector.ocr(img, det=True, rec=False)  # box only
#   recognitor = Predictor(Cfg.load_config_from_name('vgg_transformer'))
#   text = recognitor.predict(cropped_box_image)
# Đồng thời xác nhận độc lập từ 1 team SOICT 2024 (docx, không phải tự suy diễn):
#   "We employ PaddleOCR to detect text regions... For text recognition, we
#    use VietOCR, optimized specifically for Vietnamese text... Alternative
#    OCR options like Tesseract and EasyOCR were considered but proved less
#    effective for Vietnamese in complex, real-world scenes"
# Paper: DBNet++ https://arxiv.org/abs/1911.08947 · VietOCR github.com/pbcquoc/vietocr
# Viz strategy:
#   Fig A: y-position distribution histogram + region pie (200 frames)
#   Fig B: 24 frames × 3 cols [image | bbox overlay | extracted text panel]
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import cv2 as _cv2
from paddleocr import PaddleOCR
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

ocr = PaddleOCR(use_angle_cls=False, lang="vi", use_gpu=False, show_log=False)
print("[OCR] PaddleOCR 2.x loaded (CPU, detect-only)")

_vocr_cfg = Cfg.load_config_from_name("vgg_transformer")
_vocr_cfg["device"] = "cuda:0" if DEVICE == "cuda" else "cpu"
_vocr_cfg["predictor"]["beamsearch"] = True
vietocr_predictor = Predictor(_vocr_cfg)
print(f"[OCR] VietOCR (vgg_transformer) loaded on {_vocr_cfg['device']}")

def ocr_infer(img_path: str) -> list[tuple]:
    """
    Two-stage: PaddleOCR detect (det=True, rec=False, cls=False) → crop mỗi box
    → VietOCR recognize. Padding=4px quanh box — giống hệt running.py gốc.
    Returns: list of (box_4pts, text, conf)
    """
    img = _cv2.imread(img_path)
    if img is None: return []
    det_result = ocr.ocr(img_path, cls=False, det=True, rec=False)
    if not det_result or not det_result[0]: return []

    H, W = img.shape[:2]
    lines = []
    for box in det_result[0]:
        pts = [[int(p[0]), int(p[1])] for p in box]
        xs  = [p[0] for p in pts]; ys = [p[1] for p in pts]
        x0  = max(0, min(xs) - 4); y0 = max(0, min(ys) - 4)   # padding=4 như running.py
        x1  = min(W, max(xs) + 4); y1 = min(H, max(ys) + 4)
        crop = img[y0:y1, x0:x1]
        if crop.size == 0: continue
        try:
            crop_pil = Image.fromarray(_cv2.cvtColor(crop, _cv2.COLOR_BGR2RGB))
            text, prob = vietocr_predictor.predict(crop_pil, return_prob=True)
            # VietOCR translate_beam_search() (dùng khi beamsearch=True, config
            # ta set giống running.py gốc) LUÔN trả prob=None theo source code —
            # beam search decode không track sequence probability. Đây không
            # phải lỗi, là behavior đã document sẵn của VietOCR. Gán 1.0 (tin
            # cậy) để threshold conf<0.30 downstream không filter nhầm.
            if prob is None:
                prob = 1.0
        except Exception:
            continue
        if text.strip():
            lines.append((pts, text, float(prob)))
    return lines

def zone_color(y_ratio: float):
    if   y_ratio >= 0.92: return (220, 50,  50),  "ticker"
    elif y_ratio >= 0.80: return (230, 150, 30),  "subtitle"
    elif y_ratio >= 0.20: return (50,  100, 220),  "mid"
    else:                 return (50,  180, 80),   "top"

# ── A: Distribution sweep on 200 frames ──────────────────────────────────────
N_OCR_DIST = min(200, len(EVAL_FRAMES))
y_ratios, conf_values = [], []
print(f"[OCR] Distribution sweep on {N_OCR_DIST} frames...")
for fp in EVAL_FRAMES[:N_OCR_DIST]:
    for box, text, conf in ocr_infer(fp):
        if conf < 0.30: continue
        img = _cv2.imread(fp)
        if img is None: continue
        H   = img.shape[0]
        ys  = [pt[1] for pt in box]
        y_ratios.append((min(ys)+max(ys)) / 2 / H)
        conf_values.append(conf)

y_ratios = np.array(y_ratios)
print(f"[OCR] {len(y_ratios)} text detections")

region_counts = {"top_bar": 0, "mid_frame": 0, "subtitle_bar": 0, "news_ticker": 0}
for y in y_ratios:
    if   y < 0.20: region_counts["top_bar"]     += 1
    elif y < 0.80: region_counts["mid_frame"]    += 1
    elif y < 0.92: region_counts["subtitle_bar"] += 1
    else:          region_counts["news_ticker"]  += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(y_ratios, bins=60, color="steelblue", alpha=0.8, edgecolor="white")
ax1.axvline(0.80, color="orange", lw=2, ls="--", label="subtitle_bar (0.80)")
ax1.axvline(0.92, color="red",    lw=2, ls="--", label="news_ticker (0.92)")
ax1.axvspan(0.00, 0.20, alpha=0.08, color="green")
ax1.axvspan(0.20, 0.80, alpha=0.08, color="blue")
ax1.axvspan(0.80, 0.92, alpha=0.08, color="orange")
ax1.axvspan(0.92, 1.00, alpha=0.08, color="red")
ax1.set_xlabel("y_center / frame_height"); ax1.set_ylabel("Text detections")
ax1.set_title(f"OCR Text Y-Position  (n={len(y_ratios)})\nEBU R95 baseline: subtitle≥0.80, ticker≥0.92")
ax1.legend(fontsize=9)

ax2.pie(region_counts.values(), labels=region_counts.keys(), autopct="%1.1f%%",
        colors=["#32b450","#3264dc","#e69620","#dc3232"],
        wedgeprops=dict(edgecolor="white", linewidth=1.5))
ax2.set_title("Region Distribution (empirical)")
plt.tight_layout()
plt.savefig(f"{EVAL_OUT}/ocr_region_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
plt.close(fig)   # release figure memory — prevents OOM crash across sections

p90 = np.percentile(y_ratios[y_ratios > 0.70], 25) if any(y_ratios > 0.70) else 0.80
p95 = np.percentile(y_ratios[y_ratios > 0.80], 75) if any(y_ratios > 0.80) else 0.92
for k, v in region_counts.items():
    print(f"  {k:<14}: {v:4d}  ({v/max(len(y_ratios),1)*100:.1f}%)")
print(f"\n[OCR] Empirical: subtitle={p90:.2f}, ticker={p95:.2f}")
print(f"[OCR] EBU R95:   subtitle=0.80, ticker=0.92")

# ── B: 24 frames — image | bbox overlay | extracted text panel ────────────────
# 3 cols/frame để đọc trực tiếp text OCR sinh ra (đánh giá chất lượng generation),
# thay vì chỉ đọc label nhỏ đè lên box trong ảnh.
N_OCR_VIZ  = 24
VIZ_FRAMES = EVAL_FRAMES[:N_OCR_VIZ]
fig, axes = plt.subplots(N_OCR_VIZ, 3, figsize=(20, 4.2 * N_OCR_VIZ),
                          gridspec_kw={"width_ratios": [1.3, 1.3, 1]})
_axes_ocr = np.array(axes).reshape(N_OCR_VIZ, 3)

ocr_json_results = {}   # frame_name → {region: text, all_text: [...]}
REGION_COLOR_HEX = {"top_bar": "#e8f5e9", "mid_frame": "#e3f2fd",
                    "subtitle_bar": "#fff3e0", "news_ticker": "#ffebee"}

for i, fp in enumerate(VIZ_FRAMES):
    fname = os.path.basename(fp)

    # Col 0: original
    pil_orig = Image.open(fp).convert("RGB")
    _axes_ocr[i][0].imshow(np.array(pil_orig))
    _axes_ocr[i][0].set_title(fname[:35], fontsize=7)
    _axes_ocr[i][0].axis("off")

    # Col 1: bbox overlay
    img_bgr = _cv2.imread(fp)
    lines   = ocr_infer(fp)
    regions = defaultdict(list)
    if img_bgr is not None:
        H, W = img_bgr.shape[:2]
        overlay = img_bgr.copy()
        n_det = 0
        for box, text, conf in lines:
            if conf < 0.30: continue
            pts  = np.array([[int(p[0]), int(p[1])] for p in box], np.int32)
            ys   = pts[:, 1]
            y_c  = (ys.min() + ys.max()) / 2
            bgr_color, zone_name = zone_color(y_c / H)
            bgr = (bgr_color[2], bgr_color[1], bgr_color[0])
            _cv2.polylines(overlay, [pts], isClosed=True, color=bgr, thickness=2)
            mask = np.zeros_like(img_bgr)
            _cv2.fillPoly(mask, [pts], bgr)
            overlay = _cv2.addWeighted(overlay, 1.0, mask, 0.15, 0)
            label  = f"{text[:20]} {conf:.2f}"
            lx, ly = pts[0][0], max(pts[0][1] - 5, 14)
            _cv2.rectangle(overlay, (lx, ly-13), (lx + len(label)*6, ly+2), bgr, -1)
            _cv2.putText(overlay, label, (lx, ly),
                         _cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255,255,255), 1, _cv2.LINE_AA)
            n_det += 1
            zone_key = {"ticker":"news_ticker","subtitle":"subtitle_bar",
                       "mid":"mid_frame","top":"top_bar"}[zone_name]
            regions[zone_key].append(text)
        rgb_overlay = _cv2.cvtColor(overlay, _cv2.COLOR_BGR2RGB)
        _axes_ocr[i][1].imshow(rgb_overlay)
        _axes_ocr[i][1].set_title(f"{n_det} detections", fontsize=7)
    _axes_ocr[i][1].axis("off")

    # Col 2: extracted text panel — trực tiếp đọc được text OCR sinh ra per region
    region_text_out = {r: " | ".join(t) for r, t in regions.items()}
    ocr_json_results[fname] = {
        **region_text_out,
        "all_text": [t for txts in regions.values() for t in txts],
    }
    panel_lines = []
    for region_name, hexcolor in REGION_COLOR_HEX.items():
        txt = region_text_out.get(region_name, "(không có)")
        panel_lines.append(f"[{region_name}]\n{txt}")
    panel_text = "\n\n".join(panel_lines)
    wrapped = "\n".join(textwrap.wrap(panel_text, 38,
                                      replace_whitespace=False, drop_whitespace=False))
    _axes_ocr[i][2].text(0.03, 0.97, wrapped, transform=_axes_ocr[i][2].transAxes,
                         va="top", ha="left", fontsize=8,
                         bbox=dict(boxstyle="round,pad=0.4",
                                   facecolor="#fafafa", edgecolor="#bbb"))
    _axes_ocr[i][2].axis("off")

legend = [mpatches.Patch(facecolor="#32b450", label="top_bar"),
          mpatches.Patch(facecolor="#3264dc", label="mid_frame"),
          mpatches.Patch(facecolor="#e69620", label="subtitle_bar"),
          mpatches.Patch(facecolor="#dc3232", label="news_ticker")]
fig.legend(handles=legend, loc="lower center", ncol=4, fontsize=10)
plt.suptitle("PaddleOCR 2.x – 24 sample frames  [image | bbox by zone | extracted text]",
             fontsize=13)
plt.tight_layout()
plt.savefig(f"{EVAL_OUT}/ocr_bbox_viz.png", dpi=80, bbox_inches="tight")
plt.show()
plt.close(fig)   # release figure memory — prevents OOM crash across sections
save_json_results("ocr", ocr_json_results)
print("[OCR] Done")

del ocr, vietocr_predictor; gc.collect()

---

## Chạy thật — trích đặc trưng cho toàn bộ phần video của mình

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 13a – Install Feature Extraction Dependencies (chạy 1 lần)
# ═══════════════════════════════════════════════════════════════════════════════
# %%
import subprocess, sys, gc
def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("[Install-feat] Installing feature extraction dependencies...")
_pip("paddlepaddle-gpu", "paddleocr")
_pip("openai-whisper", "ffmpeg-python")
_pip("einops", "timm", "accelerate")    # Vintern dependencies
_pip("ultralytics")                      # YOLOv8
try:
    _pip("git+https://github.com/xinyu1205/recognize-anything.git", "--quiet")
    _RAM_INSTALL_OK = True
    print("[Install-feat] RAM++ installed OK")
except Exception:
    _RAM_INSTALL_OK = False
    print("[Install-feat] RAM++ not available – will use Vintern tag-prompt fallback")
_pip("transformers>=4.40.0", "faiss-gpu", "Pillow", "opencv-python-headless")
print("[Install-feat] All done\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 13b – Config + Mount + Load Progress
# ═══════════════════════════════════════════════════════════════════════════════
# %%
import os, json, glob, time, zipfile, shutil
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from PIL import Image

DRIVE_MOUNT = "/content/drive"
if not os.path.ismount(DRIVE_MOUNT):
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted.")
else:
    print("[Drive] Already mounted.")

VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
OUTPUT_ROOT  = DRIVE_OUT
INDEX_DIR    = f"{OUTPUT_ROOT}/indexes"
ZIP_DIR      = f"{OUTPUT_ROOT}/zips"
FEATURES_DIR = f"{OUTPUT_ROOT}/features"
PROGRESS_FILE= f"{OUTPUT_ROOT}/progress.json"
UNZIP_DIR    = "/content/unzip_feat"

for d in [FEATURES_DIR, UNZIP_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Hyperparams ───────────────────────────────────────────────────────────────
ASR_MODEL_SIZE  = "large-v3"   # best quality; dùng "medium" nếu cần nhanh hơn
VINTERN_MODEL   = "5CD-AI/Vintern-1B-v2"
VINTERN_BATCH   = 1            # T4: 1 an toàn nhất (2GB/frame bfloat16)
YOLO_MODEL      = "yolov8l.pt"
YOLO_CONF       = 0.30
RAM_IMG_SIZE    = 384
OCR_LANG        = "vi"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[System] Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"[System] VRAM free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB total")

def _free_gpu():
    """Release GPU memory và run garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_progress() -> dict:
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f: p = json.load(f)
    else:
        p = {}
    for key in ["done_ocr","done_asr","done_caption","done_tags","done_objects","completed_zips"]:
        p.setdefault(key, [])
    p.setdefault("features_merged", False)
    return p

def save_progress(p: dict):
    p["last_updated"] = datetime.now().isoformat()
    with open(PROGRESS_FILE, "w") as f:
        json.dump(p, f, indent=2, ensure_ascii=False)

PROGRESS = load_progress()

FULL_META_PATH = os.path.join(INDEX_DIR, "keyframe_metadata.json")
if not os.path.exists(FULL_META_PATH):
    raise FileNotFoundError("keyframe_metadata.json not found – run Part 1 first")
with open(FULL_META_PATH) as f: FULL_META = json.load(f)
VIDEO_FRAMES: dict[str, list[dict]] = defaultdict(list)
for m in FULL_META:
    VIDEO_FRAMES[m["video"]].append(m)
for v in VIDEO_FRAMES:
    VIDEO_FRAMES[v].sort(key=lambda x: x["frame_idx"])
print(f"[Meta] {len(FULL_META)} frames across {len(VIDEO_FRAMES)} videos")

# ── Helper: unzip one video from completed zips ────────────────────────────────
def unzip_video(video_name: str) -> bool:
    """Unzip frames of video_name từ completed_zips vào UNZIP_DIR. Returns True nếu found."""
    p = load_progress()
    for zip_name in p.get("completed_zips", []):
        zip_path = os.path.join(ZIP_DIR, zip_name)
        if not os.path.exists(zip_path): continue
        with zipfile.ZipFile(zip_path) as zf:
            members = [n for n in zf.namelist() if n.startswith(f"{video_name}/")]
            if not members: continue
            shutil.rmtree(os.path.join(UNZIP_DIR, video_name), ignore_errors=True)
            for m in members: zf.extract(m, UNZIP_DIR)
        return True
    return False

def rm_video_buf(video_name: str):
    shutil.rmtree(os.path.join(UNZIP_DIR, video_name), ignore_errors=True)

print(f"[Config] FEATURES_DIR: {FEATURES_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 14 – OCR  (PaddleOCR, CPU mode để không chiếm VRAM cho models sau)
# Region-based: subtitle_bar (bottom 20%) / news_ticker (bottom 8%) / mid_frame
# ─────────────────────────────────────────────────────────────────────────────
# GPU POLICY: PaddleOCR chạy trên CPU (use_gpu=False).
#   - Đủ nhanh cho text detection/recognition (10-30 frames/s trên CPU)
#   - Tiết kiệm toàn bộ VRAM cho Whisper / Vintern / RAM++ ở sections sau
#   - Load TRONG todo check → không load nếu section đã done
# ═══════════════════════════════════════════════════════════════════════════════
# %%

PROGRESS  = load_progress()
todo_ocr  = [v for v in VIDEO_FRAMES if v not in PROGRESS["done_ocr"]]

if not todo_ocr:
    print(f"[SKIP] OCR already complete ({len(PROGRESS['done_ocr'])} videos).")
else:
    print(f"[OCR] Loading PaddleOCR (CPU) – {len(todo_ocr)} videos remaining")
    from paddleocr import PaddleOCR
    import cv2 as _cv2

    # ← CPU mode: use_gpu=False giữ VRAM trống cho Whisper/Vintern sau
    ocr_engine = PaddleOCR(use_angle_cls=True, lang=OCR_LANG,
                            use_gpu=False, show_log=False)

    def _region(y_c, H):
        r = y_c / H
        if r >= 0.92: return "news_ticker"
        if r >= 0.80: return "subtitle_bar"
        if 0.20 <= r <= 0.80: return "mid_frame"
        return "top_bar"

    def ocr_frame(img_path: str) -> dict:
        res = ocr_engine.ocr(img_path, cls=True)
        if not res or not res[0]: return {}
        img = _cv2.imread(img_path)
        if img is None: return {}
        H = img.shape[0]
        regions = defaultdict(list); all_texts = []
        for line in res[0]:
            if not line: continue
            box, (text, conf) = line
            if conf < 0.45 or not text.strip(): continue
            ys  = [pt[1] for pt in box]
            y_c = (min(ys) + max(ys)) / 2
            regions[_region(y_c, H)].append(text.strip())
            all_texts.append(text.strip())
        return {
            "subtitle_bar": " | ".join(regions.get("subtitle_bar", [])),
            "news_ticker":  " | ".join(regions.get("news_ticker",  [])),
            "mid_frame":    " | ".join(regions.get("mid_frame",    [])),
            "top_bar":      " | ".join(regions.get("top_bar",      [])),
            "all_text":     all_texts,
        }

    for video_name in todo_ocr:
        out_path = os.path.join(FEATURES_DIR, f"{video_name}_ocr.json")
        if os.path.exists(out_path):
            PROGRESS["done_ocr"].append(video_name); save_progress(PROGRESS); continue
        if not unzip_video(video_name):
            print(f"  [OCR] WARN: {video_name} not in zips, skip"); continue
        frames = VIDEO_FRAMES[video_name]; t0 = time.time()
        results = {}
        for meta in frames:
            fp = os.path.join(UNZIP_DIR, video_name, meta["name"])
            if not os.path.exists(fp): continue
            r = ocr_frame(fp)
            if r: results[meta["name"]] = r
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=1)
        rm_video_buf(video_name)
        PROGRESS["done_ocr"].append(video_name); save_progress(PROGRESS)
        print(f"  [OCR] ✓ {video_name}: {len(results)} frames in {time.time()-t0:.0f}s")

    # ← Unload OCR (CPU, nhưng explicit để rõ ràng)
    del ocr_engine
    gc.collect()
    print(f"[OCR] Done. {len(PROGRESS['done_ocr'])} / {len(VIDEO_FRAMES)} videos")